# DaemonSet

Ein DaemonSet stellt sicher, dass alle (oder einige) Nodes eine Kopie eines Pods ausführen. Wenn dem Cluster Knoten hinzugefügt werden, werden ihnen Pods hinzugefügt.

Wenn Knoten aus dem Cluster entfernt werden, werden diese Pods von der Garbage Collection erfasst. Durch das Löschen eines DaemonSets werden die von ihm erstellten Pods bereinigt.

Einige typische Anwendungen eines DaemonSets sind:
* Ausführen eines Cluster-Storage-Daemons auf jedem Knoten
* Ausführen eines Daemons zum Sammeln von Protokollen auf jedem Knoten
* Ausführen eines Knotenüberwachungs-Daemons auf jedem Knoten

---

Beispiel basierend auf [DaemonSet](https://kubernetes.io/docs/concepts/workloads/controllers/daemonset/).

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: apps/v1
kind: DaemonSet
metadata:
  name: fluentd-elasticsearch
  namespace: kube-system
  labels:
    k8s-app: fluentd-logging
spec:
  selector:
    matchLabels:
      name: fluentd-elasticsearch
  template:
    metadata:
      labels:
        name: fluentd-elasticsearch
    spec:
      tolerations:
      # these tolerations are to have the daemonset runnable on control plane nodes
      # remove them if your control plane nodes should not run pods
      - key: node-role.kubernetes.io/control-plane
        operator: Exists
        effect: NoSchedule
      - key: node-role.kubernetes.io/master
        operator: Exists
        effect: NoSchedule
      containers:
      - name: fluentd-elasticsearch
        image: quay.io/fluentd_elasticsearch/fluentd:v5.0.1
        resources:
          limits:
            memory: 200Mi
          requests:
            cpu: 100m
            memory: 200Mi
        volumeMounts:
        - name: varlog
          mountPath: /var/log
      # it may be desirable to set a high priority class to ensure that a DaemonSet Pod
      # preempts running Pods
      # priorityClassName: important
      terminationGracePeriodSeconds: 30
      volumes:
      - name: varlog
        hostPath:
          path: /var/log
EOF

Es müssen, gleiche viele `fluent` Pods, wie Nodes aktiv sein. Auf jeder Node muss ein Pods laufen.    

In [ ]:
%%bash
kubectl get pods -n kube-system -o wide -l name=fluentd-elasticsearch

---
### Aufräumen

In [ ]:
%%bash
kubectl delete daemonset/fluentd-elasticsearch -n kube-system

### Links

* [Kubernetes Doku](https://kubernetes.io/docs/concepts/workloads/controllers/daemonset/)
* [Microk8s Troubleshooting](https://microk8s.io/docs/troubleshooting)
* [Manage Cluster Daemons](https://kubernetes.io/docs/tasks/manage-daemon/)